In [1]:
import sys
from pathlib import Path

# Adds the root_dir (parent of notebooks/) to sys.path
parent_dir = str(Path().resolve().parent)
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [2]:
# all libraries
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns 
from datetime import datetime
import joblib
import lightgbm as lgb 
plt.style.use('seaborn-v0_8-darkgrid')

from src.metrics import wrmsse
from src.utils import get_items_top, get_items_with_full_history

from train_and_eval import *

import warnings

In [3]:
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR/"data"/"processed"

parent_dir

'/media/ashfaque/datas/ML-projects/retail-forecast-system'

### Model stability on the future unknown datasets

* model: lgb
* training period: 2 years
* items: 373 items (top 20% in revenue)
* testing period: 137 days (in 5 windows of 28 days)

In [4]:
models_dir = BASE_DIR/'models' 

model_point = joblib.load(models_dir/"lgb_ca1_2yr_point.pkl")
model_q10 = joblib.load(models_dir/'lgb_ca1_2yr_q10.pkl')
model_q90 = joblib.load(models_dir/'lgb_ca1_2yr_q90.pkl')
models  = {'point':model_point,'q10':model_q10,'q90':model_q90}
items_373 = pd.read_pickle(models_dir/"items_full_history_ca1.pkl")['items']

future_data = pd.read_parquet(DATA_DIR/'sales_future_ca_1.parquet')
known_data = pd.read_parquet(DATA_DIR/'sales_known_ca_1.parquet')


#future data with only selected items
training_data = known_data[known_data['item_id'].isin(items_373)]
future_selected_items = future_data[future_data['item_id'].isin(items_373)]

print(f"training data: {training_data['item_id'].unique()}, \n future data: {future_selected_items['item_id'].unique()}")
# items_373

training data: ['FOODS_1_018', 'FOODS_1_024', 'FOODS_1_032', 'FOODS_1_044', 'FOODS_1_045', ..., 'HOUSEHOLD_2_450', 'HOUSEHOLD_2_465', 'HOUSEHOLD_2_483', 'HOUSEHOLD_2_509', 'HOUSEHOLD_2_514']
Length: 373
Categories (3049, object): ['FOODS_1_001', 'FOODS_1_002', 'FOODS_1_003', 'FOODS_1_004', ..., 'HOUSEHOLD_2_513', 'HOUSEHOLD_2_514', 'HOUSEHOLD_2_515', 'HOUSEHOLD_2_516'], 
 future data: ['FOODS_1_018', 'FOODS_1_024', 'FOODS_1_032', 'FOODS_1_044', 'FOODS_1_045', ..., 'HOUSEHOLD_2_450', 'HOUSEHOLD_2_465', 'HOUSEHOLD_2_483', 'HOUSEHOLD_2_509', 'HOUSEHOLD_2_514']
Length: 373
Categories (3049, object): ['FOODS_1_001', 'FOODS_1_002', 'FOODS_1_003', 'FOODS_1_004', ..., 'HOUSEHOLD_2_513', 'HOUSEHOLD_2_514', 'HOUSEHOLD_2_515', 'HOUSEHOLD_2_516']


In [5]:
min_date = training_data['date'].min()
max_date = training_data['date'].max()
print(max_date,min_date,(max_date-min_date).days)

2016-01-05 00:00:00 2011-01-29 00:00:00 1802


In [6]:
cat_categories = {}
CAT_COLS = ['item_id', 'dept_id', 'cat_id']
for col in CAT_COLS:
    cats = sorted(training_data[col].dropna().unique().tolist())
    training_data[col] = pd.Categorical(training_data[col], categories=cats)
    cat_categories[col] = cats
# cat_categories

In [7]:
future_selected_items.columns

Index(['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd',
       'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year',
       'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2',
       'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'day_of_week',
       'day_of_month', 'week_of_year'],
      dtype='object')

* to figure out why wrmsse jumped too much across windows, lets, retrain our model after every month 

In [10]:
training_data.columns

Index(['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd',
       'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year',
       'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2',
       'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'day_of_week',
       'day_of_month', 'week_of_year'],
      dtype='object')

In [15]:
from src.pipeline import recursive_forecast_batch
from src.features import get_known_future_features, get_lag_rolling_features


FEATURE_COLS = ['lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_28','day_of_week', 'day_of_month',
                 'week_of_year', 'month', 'year', 'sell_price', 'item_id', 'dept_id', 'cat_id']


static_feature_cols = ['day_of_week', 'day_of_month','week_of_year', 'month', 'year', 'sell_price', 'item_id', 'dept_id', 'cat_id']

raw_history_cols = ['date','sales'] + static_feature_cols
# Ensure future_selected_items contains static features prejoind (sell_price, dept_id, cat_id)
# Base raw sales history (3 columns only)

cutoff = training_data['date'].max()-pd.Timedelta(days=730) # fixed (do not change training period, wrmsse will change)
raw_history = training_data[training_data['date']>=cutoff][raw_history_cols].copy()
fixed_training_base = raw_history.copy()

future_sales = future_selected_items.copy()
future_start = future_sales['date'].min()
future_total_days = (future_sales['date'].max()-future_start).days
num_windows = int(future_total_days/28) 

# select mode: True = Retrain models every month; False = Use static model (previously trained on 2 years of data)

ENABLE_RETRAINING = False

active_models = models  # Initial training models
window_scores = {}

# code here 

for i in range(num_windows+1):
    future_end = future_start + pd.Timedelta(days=28)
    if future_end > future_sales['date'].max():
        future_end = future_sales['date'].max()

    mask = (future_sales['date']>= future_start) & (future_sales['date']< future_end)
    future_window_static = future_sales[mask].copy() # copying each window

    print(f"\n=== Window {i+1}: {future_start.date()} to {future_end.date()} ===")

    # 1. Generate recursive forecasts for the 28-day window
    pred_window = recursive_forecast_batch(models=active_models,history_df=raw_history,future_static_df=future_window_static,
                                           feature_cols=FEATURE_COLS,cat_categories=cat_categories)

    # 2. score predictions against actual ground truth
    score_window = wrmsse(fixed_training_base, future_window_static, pred_window) # fixed training is used for fair comparison, else
                                                                                   # wrmsse would change
    window_scores[f'window_{i+1}'] = score_window
    print(f"\n WRMSSE: {score_window}")
    # 3. post evaluation: if using static or retraining
    

    if ENABLE_RETRAINING:
        
        #  1. select static features cols from future window
        actuals = future_window_static[raw_history.columns].copy()

        # make sure that actuals and raw_history has exact same columns
        assert set(actuals.columns) == set(raw_history.columns), (f"Column mismatch!\n"f"Actual columns: {list(actuals.columns)}\n\
                                                                  Raw history columns: {list(raw_history.columns)}")
        # Append predicted actuals to raw sales history for retraining
        raw_history = pd.concat([raw_history,actuals],ignore_index=True)
        # build lag and rolling features for this new training data
        new_train_features = get_lag_rolling_features(raw_history)

        # retrain and update the models
        retrained_models, _,_ = train_models(new_train_features)
        active_models = retrained_models

        
    else:
        # static model mode: update history with PREDICTIONS to feed lags into window i+1
        pred_to_append = future_window_static[static_feature_cols+['date']].copy() # to merge with raw history, need exact columns
        pred_to_append = pred_to_append.merge(pred_window[['item_id','date','sales_pred']],on=['item_id','date'],
                                              how='inner').rename(columns={'sales_pred':'sales'})

        # align the columns with raw_history
        pred_to_append = pred_to_append[raw_history.columns]

        # append predictions to  raw_history so lag_7, rolling_mean_28 use forecasted values not real actual values
        raw_history = pd.concat([raw_history,pred_to_append],ignore_index=True)
        
    # 4. reset the start of future window
    future_start = future_end


=== Window 1: 2016-01-06 to 2016-02-03 ===

 WRMSSE: 0.7528060478676313

=== Window 2: 2016-02-03 to 2016-03-02 ===

 WRMSSE: 0.8040666537874086

=== Window 3: 2016-03-02 to 2016-03-30 ===

 WRMSSE: 0.8035458619281157

=== Window 4: 2016-03-30 to 2016-04-27 ===

 WRMSSE: 0.7936692472078208

=== Window 5: 2016-04-27 to 2016-05-22 ===

 WRMSSE: 0.8317588564156941


In [19]:
from src.pipeline import recursive_forecast_batch
from src.features import get_known_future_features, get_lag_rolling_features


FEATURE_COLS = ['lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_28','day_of_week', 'day_of_month',
                 'week_of_year', 'month', 'year', 'sell_price', 'item_id', 'dept_id', 'cat_id']


static_feature_cols = ['day_of_week', 'day_of_month','week_of_year', 'month', 'year', 'sell_price', 'item_id', 'dept_id', 'cat_id']

raw_history_cols = ['date','sales'] + static_feature_cols
# Ensure future_selected_items contains static features prejoind (sell_price, dept_id, cat_id)
# Base raw sales history (3 columns only)

cutoff = training_data['date'].max()-pd.Timedelta(days=730) # fixed (do not change training period, wrmsse will change)
raw_history = training_data[training_data['date']>=cutoff][raw_history_cols].copy()
fixed_training_base = raw_history.copy()

future_sales = future_selected_items.copy()
future_start = future_sales['date'].min()
future_total_days = (future_sales['date'].max()-future_start).days
num_windows = int(future_total_days/28) 

# select mode: True = Retrain models every month; False = Use static model (previously trained on 2 years of data)

ENABLE_RETRAINING = True
if ENABLE_RETRAINING:
    print("Retraining every 28 days")

active_models = models  # Initial training models
window_scores = {}

# code here 

for i in range(num_windows+1):
    future_end = future_start + pd.Timedelta(days=28)
    if future_end > future_sales['date'].max():
        future_end = future_sales['date'].max()

    mask = (future_sales['date']>= future_start) & (future_sales['date']< future_end)
    future_window_static = future_sales[mask].copy() # copying each window

    print(f"\n=== Window {i+1}: {future_start.date()} to {future_end.date()} ===")

    # 1. Generate recursive forecasts for the 28-day window
    pred_window = recursive_forecast_batch(models=active_models,history_df=raw_history,future_static_df=future_window_static,
                                           feature_cols=FEATURE_COLS,cat_categories=cat_categories)

    # 2. score predictions against actual ground truth
    score_window = wrmsse(fixed_training_base, future_window_static, pred_window) # fixed training is used for fair comparison, else
                                                                                   # wrmsse would change
    window_scores[f'window_{i+1}'] = score_window
    print(f"\n WRMSSE: {score_window}")
    # 3. post evaluation: if using static or retraining
    

    if ENABLE_RETRAINING:
        
        #  1. select static features cols from future window
        actuals = future_window_static[raw_history.columns].copy()

        # make sure that actuals and raw_history has exact same columns
        assert set(actuals.columns) == set(raw_history.columns), (f"Column mismatch!\n"f"Actual columns: {list(actuals.columns)}\n\
                                                                  Raw history columns: {list(raw_history.columns)}")
        # Append predicted actuals to raw sales history for retraining
        raw_history = pd.concat([raw_history,actuals],ignore_index=True)
        # build lag and rolling features for this new training data
        new_train_features = get_lag_rolling_features(raw_history)

        # retrain and update the models
        retrained_models, _,_ = train_models(new_train_features)
        active_models = retrained_models

        
    else:
        # static model mode: update history with PREDICTIONS to feed lags into window i+1
        pred_to_append = future_window_static[static_feature_cols+['date']].copy() # to merge with raw history, need exact columns
        pred_to_append = pred_to_append.merge(pred_window[['item_id','date','sales_pred']],on=['item_id','date'],
                                              how='inner').rename(columns={'sales_pred':'sales'})

        # align the columns with raw_history
        pred_to_append = pred_to_append[raw_history.columns]

        # append predictions to  raw_history so lag_7, rolling_mean_28 use forecasted values not real actual values
        raw_history = pd.concat([raw_history,pred_to_append],ignore_index=True)
        
    # 4. reset the start of future window
    future_start = future_end

Retraining every 28 days

=== Window 1: 2016-01-06 to 2016-02-03 ===

 WRMSSE: 0.7528060478676313

=== Window 2: 2016-02-03 to 2016-03-02 ===

 WRMSSE: 0.7690188165728933

=== Window 3: 2016-03-02 to 2016-03-30 ===

 WRMSSE: 0.7548656033874784

=== Window 4: 2016-03-30 to 2016-04-27 ===

 WRMSSE: 0.7675054883916502

=== Window 5: 2016-04-27 to 2016-05-22 ===

 WRMSSE: 0.7765869601935307


* After retraining every 4 weeks, we can see that the WRMSSE is stable across 4 windows, which signals that previously the inflation in WRMSSE
  was due to recursive error accumulating over the period, not model drift, not underlying pattern change.

* So the current model is good as when horizon is maximum 4 weeks. 